# M7 Grad-CAM - interpretability (pre-registered criteria)

Loads the saved M7-v1 model (`models/M7_run7/gradcam_model.pt`), computes 1D Grad-CAM over time for fold9 WPW true-positives, aligns saliency to detected R-peaks, and reads **where** the model looks.

> **Pre-registered verdict:** onset/QRS-dominant -> confirms delta physiology; ST/T-dominant -> potential orthogonality (cross with rho); baseline/edges-dominant -> artifact red flag (cross with dataset confound). Cheap (~1 min, fold9 only).

### Block g0: Load model + fold9 + select WPW true-positives

In [ ]:
# M7 Grad-CAM - interpretability of M7-v1 with PRE-REGISTERED criteria.
# Loads the saved classifier (Run 7 gradcam_model.pt), computes 1D Grad-CAM saliency over time for fold9 WPW
# true-positives, aligns it to detected beats, and reads WHERE the model looks vs the cardiac cycle.
import os, sys, json, warnings
import numpy as np
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
ROOT=r'.'
PROCESSED=os.path.join(ROOT,'data','processed'); SRC=os.path.join(ROOT,'src')
FIG=os.path.join(ROOT,'reports','figures'); METRICS=os.path.join(ROOT,'reports','metrics')
MODELS=os.path.join(ROOT,'models','M7_run7'); os.makedirs(FIG,exist_ok=True)
sys.path.insert(0,SRC); from signal_loading import LEADS_CANONICAL
import torch, torch.nn as nn
class BasicBlock1d(nn.Module):
    def __init__(self,cin,cout,stride=1):
        super().__init__()
        self.conv1=nn.Conv1d(cin,cout,7,stride=stride,padding=3,bias=False); self.bn1=nn.BatchNorm1d(cout)
        self.conv2=nn.Conv1d(cout,cout,7,padding=3,bias=False); self.bn2=nn.BatchNorm1d(cout)
        self.relu=nn.ReLU(inplace=True); self.down=None
        if stride!=1 or cin!=cout:
            self.down=nn.Sequential(nn.Conv1d(cin,cout,1,stride=stride,bias=False),nn.BatchNorm1d(cout))
    def forward(self,x):
        idt=x if self.down is None else self.down(x)
        o=self.relu(self.bn1(self.conv1(x))); o=self.bn2(self.conv2(o)); return self.relu(o+idt)
class ResNet1d(nn.Module):
    def __init__(self,ch=(16,32,64),pdrop=0.3):
        super().__init__()
        self.stem=nn.Sequential(nn.Conv1d(12,ch[0],15,stride=2,padding=7,bias=False),
                                nn.BatchNorm1d(ch[0]),nn.ReLU(inplace=True),nn.MaxPool1d(4))
        self.layer1=BasicBlock1d(ch[0],ch[0],1); self.layer2=BasicBlock1d(ch[0],ch[1],2); self.layer3=BasicBlock1d(ch[1],ch[2],2)
        self.pool=nn.AdaptiveMaxPool1d(1); self.head=nn.Sequential(nn.Flatten(),nn.Dropout(pdrop),nn.Linear(ch[2],1))
    def forward(self,x):
        return self.head(self.pool(self.layer3(self.layer2(self.layer1(self.stem(x)))))).squeeze(1)
model=ResNet1d(); model.load_state_dict(torch.load(os.path.join(MODELS,'gradcam_model.pt')),strict=False); model.eval()
z=np.load(os.path.join(PROCESSED,'m7_run2_cache','m7_run2_data.npz'),allow_pickle=True)
X9=np.asarray(z['X9']); y9=z['y9']
with torch.no_grad():
    sc=[torch.sigmoid(model(torch.tensor(np.ascontiguousarray(X9[i:i+256],dtype=np.float32)))).numpy() for i in range(0,len(X9),256)]
scores=np.concatenate(sc)
wpw=np.where(y9==1)[0]; sel=wpw[np.argsort(-scores[wpw])][:6]   # top-scored WPW true-positives
FS=500; LEAD_II=LEADS_CANONICAL.index('II')
print('fold9 WPW %d | selected top %d (scores %s)'%(len(wpw),len(sel),np.round(scores[sel],3)))
print('Block g0 OK.')


### Block g1: 1D Grad-CAM (last residual block)

In [ ]:
# 1D Grad-CAM on the last residual block (layer3): gradient-weighted activations -> temporal saliency.
acts={}; grads={}
model.layer3.register_forward_hook(lambda m,i,o: acts.__setitem__('v',o))
model.layer3.register_full_backward_hook(lambda m,gi,go: grads.__setitem__('v',go[0].detach()))
def gradcam(xrow):
    x=torch.tensor(np.ascontiguousarray(xrow[None],dtype=np.float32),requires_grad=False)
    model.zero_grad(); logit=model(x); logit.backward()
    A=acts['v'][0].detach(); G=grads['v'][0]                  # (64, T')
    w=G.mean(dim=1)                                           # channel weights = GAP of gradients
    cam=torch.relu((w[:,None]*A).sum(0))                      # (T',)
    cam=cam/(cam.max()+1e-8)
    cam=torch.nn.functional.interpolate(cam[None,None],size=xrow.shape[1],mode='linear',align_corners=False)[0,0]
    return cam.detach().numpy(), float(torch.sigmoid(logit))
print('Block g1 OK.')


### Block g2: Beat alignment + windowed saliency

In [ ]:
# Beat alignment: detect R-peaks (lead II), aggregate saliency in windows relative to R (PRE-REGISTERED).
# Windows (ms @500Hz): ONSET/delta [-80,+10], QRS [-30,+50], ST-T [+150,+400], BASELINE [-320,-140].
WIN={'onset':(-80,10),'qrs':(-30,50),'st_t':(150,400),'baseline':(-320,-140)}
def ms(a): return int(a*FS/1000)
def beat_windows(cam, rpeaks):
    out={k:[] for k in WIN}
    for r in rpeaks:
        for k,(a,b) in WIN.items():
            i0,i1=r+ms(a),r+ms(b)
            if 0<=i0<i1<=len(cam): out[k].append(cam[i0:i1].mean())
    return {k:(np.mean(v) if v else np.nan) for k,v in out.items()}
agg={k:[] for k in WIN}; cams=[]; rpk=[]
for idx in sel:
    cam,p=gradcam(X9[idx]); cams.append(cam)
    sigII=X9[idx,LEAD_II]; rp,_=find_peaks(sigII,distance=int(0.4*FS),prominence=1.0); rpk.append(rp)
    bw=beat_windows(cam,rp)
    for k in WIN: agg[k].append(bw[k])
prof={k:float(np.nanmean(agg[k])) for k in WIN}
print('Aggregate beat-aligned saliency (mean over %d WPW):'%len(sel))
for k in ['onset','qrs','st_t','baseline']: print('   %-9s %.3f'%(k,prof[k]))
print('Block g2 OK.')


### Block g3: Figures + pre-registered verdict

In [ ]:
# Figure + PRE-REGISTERED verdict: onset/QRS-dominant -> confirms delta physiology;
# ST-T-dominant -> potential orthogonality (cross with rho); baseline/edges -> artifact red flag (cross with dataset confound).
fig,ax=plt.subplots(3,2,figsize=(15,10))
for j,idx in enumerate(sel):
    a=ax[j//2,j%2]; sig=X9[idx,LEAD_II]; cam=cams[j]; t=np.arange(len(sig))/FS
    a.plot(t,sig,color='#1e293b',lw=0.6)
    a.fill_between(t,sig.min(),sig.max(),where=cam>0.5,color='#ef4444',alpha=0.18,step='mid')
    a.scatter(rpk[j]/FS,sig[rpk[j]],c='#2563eb',s=12,zorder=5)
    a.set_title('WPW #%d  score %.2f  (lead II, red = high Grad-CAM)'%(idx,scores[idx]),fontsize=9)
    a.set_xlabel('s'); a.set_xlim(0,5)
plt.suptitle('M7-v1 Grad-CAM on fold9 WPW true-positives (lead II)'); plt.tight_layout()
p1=os.path.join(FIG,'M7_gradcam_examples.png'); plt.savefig(p1,dpi=140,bbox_inches='tight'); plt.show()

fig2,ax2=plt.subplots(figsize=(7,4))
ks=['baseline','onset','qrs','st_t']; vals=[prof[k] for k in ks]
ax2.bar(ks,vals,color=['#94a3b8','#ef4444','#f59e0b','#3b82f6'])
ax2.set_ylabel('mean Grad-CAM saliency'); ax2.set_title('Beat-aligned saliency by cardiac-cycle window (pre-registered criteria)')
p2=os.path.join(FIG,'M7_gradcam_beataligned.png'); plt.savefig(p2,dpi=140,bbox_inches='tight'); plt.show()

ranked=sorted(prof,key=lambda k:-prof[k]); top=ranked[0]
verdict={'onset':'CONFIRMS delta/QRS-onset physiology',
         'qrs':'QRS-dominant (morphology) - plausible, cross-check with rho',
         'st_t':'ST/T-dominant -> potential orthogonality, cross with rho(M3,M4)',
         'baseline':'BASELINE-dominant -> ARTIFACT red flag, cross with dataset confound'}[top]
print('=== Grad-CAM verdict ===')
print('  dominant window: %s (%.3f) -> %s'%(top,prof[top],verdict))
json.dump({'profile':prof,'dominant':top,'verdict':verdict,'n_wpw':len(sel)},
          open(os.path.join(METRICS,'m7_gradcam.json'),'w'),indent=2)
print('  figures: %s | %s'%(p1,p2)); print('Block g3 OK.')
